# Demo 04. Numerical Differentiation and the Optimal Step Size

**Module 2** (numerical differentiation), the step-size study of section 2.4.

A finite-difference derivative carries two competing errors. Truncation error comes from replacing the limit $h\to0$ with a fixed $h$ and shrinks as $h$ decreases. Rounding error comes from subtracting two nearly equal function values and *grows* as $h$ decreases. Their sum is a U-shaped curve with a finite optimal step size. This demo measures that curve against a high-precision reference and confirms the predicted location of its minimum.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp
from ipywidgets import interact, Dropdown, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)
mp.mp.dps = 50   # 50-digit reference for the exact derivative

## The two error terms

For the forward difference $D_+f(x)=\dfrac{f(x+h)-f(x)}{h}$, Taylor expansion gives a truncation error $\tfrac{h}{2}f''(\xi)$, while the stored function values each carry absolute rounding error near $u\lvert f\rvert$, where $u\approx1.1\times10^{-16}$ is the unit roundoff. The subtraction divides by $h$, so the rounding contribution is about $2u\lvert f\rvert/h$. The total error model is

$$ E_+(h) \approx \tfrac{h}{2}\lvert f''\rvert + \frac{2u\lvert f\rvert}{h}, \qquad h_\* \approx 2\sqrt{\frac{u\lvert f\rvert}{\lvert f''\rvert}} = O(\sqrt u)\approx 10^{-8}. $$

The central difference $D_0 f(x)=\dfrac{f(x+h)-f(x-h)}{2h}$ has truncation error $\tfrac{h^2}{6}f'''$, so

$$ E_0(h) \approx \tfrac{h^2}{6}\lvert f'''\rvert + \frac{u\lvert f\rvert}{h}, \qquad h_\* \approx \left(\frac{3u\lvert f\rvert}{\lvert f'''\rvert}\right)^{1/3} = O(u^{1/3})\approx 10^{-5}. $$

The central formula reaches a smaller minimum error at a larger step size.

In [ ]:
# ---------------------------------------------------------------------------
# Test functions and their exact derivatives at 50-digit precision
# ---------------------------------------------------------------------------
# We differentiate in ordinary float64 but compare against a reference value
# computed by mpmath, so the measured error is the true error of the float
# computation, not an artifact of an approximate "exact" answer.

FUNCS = {
    "exp(x)":        (np.exp,               mp.e),           # f'(1) = e
    "sin(x)":        (np.sin,               mp.cos),         # f'(x) = cos x
    "1/(1+x^2)":     (lambda x: 1.0/(1.0+x*x), None),        # handled below
}

def exact_derivative(name, x0):
    """Reference value f'(x0) from mpmath at 50 digits."""
    if name == "exp(x)":
        return float(mp.e ** mp.mpf(x0))
    if name == "sin(x)":
        return float(mp.cos(mp.mpf(x0)))
    if name == "1/(1+x^2)":
        f = lambda t: 1 / (1 + t * t)
        return float(mp.diff(f, mp.mpf(x0)))   # mpmath differentiates numerically at high dps
    raise KeyError(name)

In [ ]:
# ---------------------------------------------------------------------------
# The two finite-difference operators
# ---------------------------------------------------------------------------
def forward_diff(f, x, h):
    """Forward difference (f(x+h) - f(x)) / h. First-order accurate."""
    return (f(x + h) - f(x)) / h

def central_diff(f, x, h):
    """Central difference (f(x+h) - f(x-h)) / (2h). Second-order accurate."""
    return (f(x + h) - f(x - h)) / (2.0 * h)

In [ ]:
# ---------------------------------------------------------------------------
# Sweep h over many orders of magnitude and plot the U-curve
# ---------------------------------------------------------------------------
# Each branch of the U is a straight line on log-log axes: the truncation
# branch has the slope of the method's order, and the rounding branch has
# slope -1 (both formulas divide a fixed rounding floor by h).

U = np.finfo(float).eps    # float64 unit roundoff, ~2.2e-16

def show_ucurve(func="exp(x)", x0=1.0):
    """Plot total error vs step size for both formulas and mark the minima."""
    f, _ = FUNCS[func]
    exact = exact_derivative(func, x0)
    hs = np.logspace(-16, 0, 400)

    err_fwd = np.abs(np.array([forward_diff(f, x0, h) for h in hs]) - exact)
    err_cen = np.abs(np.array([central_diff(f, x0, h) for h in hs]) - exact)

    plt.figure()
    plt.loglog(hs, err_fwd + 1e-20, "r.-", ms=3, label="forward  O(h)")
    plt.loglog(hs, err_cen + 1e-20, "b.-", ms=3, label="central  O(h^2)")
    # predicted optimal step sizes
    plt.axvline(np.sqrt(U),      color="r", ls=":", lw=1, label="h* ~ sqrt(u)")
    plt.axvline(U ** (1.0/3.0),  color="b", ls=":", lw=1, label="h* ~ u^(1/3)")
    plt.xlabel("step size h"); plt.ylabel("absolute error")
    plt.title(f"Error vs step size for d/dx {func} at x={x0}")
    plt.legend(); plt.show()

    print(f"forward: min error {err_fwd.min():.2e} at h = {hs[err_fwd.argmin()]:.2e}"
          f"   (predicted h* ~ {np.sqrt(U):.2e})")
    print(f"central: min error {err_cen.min():.2e} at h = {hs[err_cen.argmin()]:.2e}"
          f"   (predicted h* ~ {U**(1/3):.2e})")

show_ucurve()

## Richardson extrapolation

The central difference has an error expansion in even powers of $h$:
$D_0 f(x) = f'(x) + c_2 h^2 + c_4 h^4 + \cdots$. Evaluating at $h$ and $h/2$ and forming
$\tfrac{4 D_0(h/2) - D_0(h)}{3}$ cancels the $h^2$ term and leaves an $O(h^4)$ approximation. This buys two extra orders from two evaluations, without pushing $h$ into the rounding-dominated regime.

In [ ]:
# ---------------------------------------------------------------------------
# Richardson extrapolation of the central difference: O(h^2) -> O(h^4)
# ---------------------------------------------------------------------------
def richardson(f, x, h):
    """Combine central differences at h and h/2 to cancel the leading h^2 term."""
    D_h  = central_diff(f, x, h)
    D_h2 = central_diff(f, x, h / 2.0)
    return (4.0 * D_h2 - D_h) / 3.0

def show_richardson(func="exp(x)", x0=1.0, log10_h=-2.0):
    """Compare central difference and its Richardson extrapolation at one h."""
    f, _ = FUNCS[func]
    exact = exact_derivative(func, x0)
    h = 10.0 ** log10_h
    e_cen  = abs(central_diff(f, x0, h) - exact)
    e_rich = abs(richardson(f, x0, h)  - exact)
    print(f"h = {h:.1e}")
    print(f"  central     error : {e_cen:.3e}   (order 2)")
    print(f"  Richardson  error : {e_rich:.3e}   (order 4)")
    print(f"  improvement factor: {e_cen / max(e_rich, 1e-300):.1f}x")

show_richardson()

In [ ]:
# ---------------------------------------------------------------------------
# Interactive controls
# ---------------------------------------------------------------------------
interact(
    show_ucurve,
    func=Dropdown(options=list(FUNCS), description="f"),
    x0=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description="x0"),
);
interact(
    show_richardson,
    func=Dropdown(options=list(FUNCS), description="f"),
    x0=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description="x0"),
    log10_h=FloatSlider(value=-2.0, min=-6.0, max=-0.5, step=0.5, description="log10(h)"),
);

## Summary

- Finite-difference error is a sum of a truncation term that falls with $h$ and a rounding term that rises as $h\to0$, giving a finite optimal step size.
- The forward difference minimizes near $h_\*\sim\sqrt u\approx10^{-8}$; the central difference minimizes near $h_\*\sim u^{1/3}\approx10^{-5}$ with a smaller minimum error.
- Shrinking $h$ past the minimum makes the answer worse, not better. This is the roundoff-versus-truncation tradeoff in its clearest form.
- Richardson extrapolation raises the order from two to four using one extra evaluation, avoiding the rounding-dominated regime.